In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from src.data import *
from src.post_process_functions import *
from src.data_format import format_output, LIST_COLS

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [49]:
cols_to_open = ["uid", "disasterType", "appealCode", "reportDate", "extraction_error","processing_error"]
dropped_df = pd.read_parquet(DATA_IN_JSONS / "no_text_dropped_preproc_text_sel_gaps_merged_df_with_clean_text_all-all_reports_with_pdf_status_020726_with_clean_text_v090726.parquet") 
ymin = 2016
ymax = 2025
dropped_df["reportYear"] = pd.to_datetime(dropped_df["date"], errors="coerce", infer_datetime_format=True).dt.year
dropped_df = dropped_df.loc[(dropped_df["reportYear"]>=ymin) & (dropped_df["reportYear"]<=ymax)]

# final_df = pd.read_parquet(DATA_IN_JSONS / "no_text_preproc_text_sel_gaps_merged_df_with_clean_text_all-all_reports_with_pdf_status_020726_with_clean_text_v090726.parquet")
final_df = pd.read_csv(DATA_IN_JSONS / "preproc_text_sel_gaps_df_with_clean_text_all_combined_v300626_v060726.csv")

C:\Users\lhasbini\AppData\Local\Temp\ipykernel_10436\1431537588.py:5: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dropped_df["reportYear"] = pd.to_datetime(dropped_df["date"], errors="coerce", infer_datetime_format=True).dt.year
C:\Users\lhasbini\AppData\Local\Temp\ipykernel_10436\1431537588.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dropped_df["reportYear"] = pd.to_datetime(dropped_df["date"], errors="coerce", infer_datetime_format=True).dt.year


In [50]:
dropped_df.processing_error.value_counts()

processing_error
invalid appealType    3290
no natural hazard     1176
not in English           3
no text                  3
Name: count, dtype: int64

In [74]:
dropped_df.loc[dropped_df["processing_error"]!="invalid appealType"].appealCode.nunique()

474

In [70]:
dropped_df.appealCode.nunique()

1716

In [71]:
dropped_df.loc[dropped_df["processing_error"]=="invalid appealType"].appealCode.nunique()

1529

In [69]:
dropped_df.loc[dropped_df["processing_error"]=="no natural hazard"].appealCode.nunique()

471

In [ ]:
print("Total number of unique reports ", len(final_df)+len(dropped_df)-len(dropped_df.loc[dropped_df["processing_error"].isin(["invalid appealType", "missing origType"])]))
print("Total number of unique appealCodes ", pd.concat([final_df["appealCode"], dropped_df.loc[dropped_df["processing_error"]!="invalid appealType"]["appealCode"]], ignore_index=True).nunique())


Total number of unique reports  5379
Total number of unique appealCodes  1851


In [65]:
# Find the number of unique appealCodes which are dropped because of "no natural hazard" and are not present anymore in the final_df
dropped_no_natural_hazard = dropped_df[
    (dropped_df["processing_error"] == "invalid appealType") &
    (~dropped_df["appealCode"].isin(final_df["appealCode"]))
]

reports_correct_appeal = pd.concat([final_df, dropped_df], ignore_index=True)
reports_correct_appeal = reports_correct_appeal.loc[~reports_correct_appeal["appealCode"].isin(dropped_no_natural_hazard["appealCode"])]
print(f"Number of unique reports {reports_correct_appeal.shape[0]} after dropping reports with invalid appealType")
print(f"Number of unique appeals {reports_correct_appeal['appealCode'].nunique()} after dropping reports with invalid appealType")
print(
    "Number of unique appealCodes dropped because of 'invalid appealType' and not present in final_df:",
    dropped_no_natural_hazard["appealCode"].nunique()
)

Number of unique reports 6107 after dropping reports with invalid appealType
Number of unique appeals 1589 after dropping reports with invalid appealType
Number of unique appealCodes dropped because of 'invalid appealType' and not present in final_df: 1009


In [64]:
# Find the number of unique appealCodes which are dropped because of "no natural hazard" and are not present anymore in the final_df
# dropped_no_natural_hazard = reports_correct_appeal[
#     (reports_correct_appeal["processing_error"] == "no natural hazard") &
#     (~reports_correct_appeal["appealCode"].isin(final_df["appealCode"]))
# ]
dropped_no_natural_hazard = dropped_df[
    (dropped_df["processing_error"] == "no natural hazard") &
    (~dropped_df["appealCode"].isin(final_df["appealCode"]))
]

print(
    "Number of reports appealCodes dropped because of 'no natural hazard' and not present in final_df:",
    dropped_no_natural_hazard.shape[0]
)
print(
    "Number of unique appealCodes dropped because of 'no natural hazard' and not present in final_df:",
    dropped_no_natural_hazard["appealCode"].nunique()
)

Number of reports appealCodes dropped because of 'no natural hazard' and not present in final_df: 1100
Number of unique appealCodes dropped because of 'no natural hazard' and not present in final_df: 442


In [ ]:
# Find the number of unique appealCodes which are dropped because of "not in English" and are not present anymore in the final_df
dropped_no_english = dropped_df[
    (dropped_df["processing_error"] == "not in English") &
    (~dropped_df["appealCode"].isin(final_df["appealCode"]))
]

print(
    "Number of reports appealCodes dropped because of 'not in English' and not present in final_df:",
    dropped_no_english.shape[0]
)

Number of unique appealCodes dropped because of 'not in English' and not present in final_df: 0


In [66]:
# Find the number of unique appealCodes which are dropped because of "no impact text" and are not present anymore in the final_df
dropped_no_impact_text = dropped_df[
    (dropped_df["processing_error"] == "no impact text") &
    (~dropped_df["appealCode"].isin(final_df["appealCode"]))
]

print(
    "Number of unique appealCodes dropped because of 'no impact text' and not present in final_df:",
    dropped_no_impact_text["appealCode"].nunique()
)

Number of unique appealCodes dropped because of 'no impact text' and not present in final_df: 0


In [67]:
# Find the number of unique appealCodes which are dropped because of "no impact text" and are not present anymore in the final_df
dropped_no_text = dropped_df[
    (dropped_df["processing_error"] == "no impact text") &
    (~dropped_df["appealCode"].isin(final_df["appealCode"]))
]
print(
    "Number of unique reports dropped because of 'no text' and not present in final_df:",
    dropped_no_text.shape[0]
)
print(
    "Number of unique appealCodes dropped because of 'no text' and not present in final_df:",
    dropped_no_text["appealCode"].nunique()
)

Number of unique reports dropped because of 'no text' and not present in final_df: 0
Number of unique appealCodes dropped because of 'no text' and not present in final_df: 0


In [32]:
final_df["appealCode"].nunique()

1410